# Intelligent Library Chat Assistant - Step-by-Step Demo

## Chapter 3: Architecture & Evaluation

This notebook demonstrates the **inner workings** of our Intelligent Library Chat Assistant system. 
Following the architecture from Chapter 3, we show each component's input, processing, and output.

### System Architecture Overview:
1. **Input Processing** -> Text preprocessing & tokenization
2. **Feature Extraction** -> Bag-of-Words & TF-IDF
3. **Intent Classification** -> ML-based classification
4. **Rule Engine** -> Pattern matching for specific queries
5. **Hybrid Integration** -> Combining ML + Rules
6. **Response Generation** -> Template-based responses

Let's begin!


## Cell 1: Environment Setup & Library Versions

In [1]:
# Import required libraries
import sys
import json
import re
from collections import Counter

# NLP Libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML Libraries
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn import __version__ as sklearn_version

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Display versions
print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)
print(f"Python Version: {sys.version}")
print(f"NLTK Version: {nltk.__version__}")
print(f"Sklearn Version: {sklearn_version}")
print("\nAll libraries loaded successfully!")


ENVIRONMENT SETUP
Python Version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
NLTK Version: 3.9.1
Sklearn Version: 1.6.1

All libraries loaded successfully!


## Cell 2: Sample Training Data Creation

In [2]:
# Create sample training data for the library chatbot
training_data = [
    # Book Search Intent
    ("I want to find a book about Python programming", "book_search"),
    ("Can you search for books by author J.K. Rowling", "book_search"),
    ("Find me books on machine learning", "book_search"),
    ("Search for the novel 1984", "book_search"),
    ("I am looking for science fiction books", "book_search"),
    ("Do you have any books about web development", "book_search"),
    ("Find books on data science", "book_search"),
    ("Search library catalog for Harry Potter", "book_search"),
    
    # Availability Intent
    ("Is the book Introduction to Algorithms available", "availability"),
    ("Can I borrow Clean Code today", "availability"),
    ("Is The Great Gatsby in stock", "availability"),
    ("Are there copies of Harry Potter available", "availability"),
    ("Check if Design Patterns is available", "availability"),
    ("I want to know if 1984 is available for checkout", "availability"),
    ("Is there a waiting list for this book", "availability"),
    ("When will The Hobbit be available", "availability"),
    
    # Borrow Intent
    ("How do I borrow a book", "borrow"),
    ("I want to check out this book", "borrow"),
    ("Can I reserve The Catcher in the Rye", "borrow"),
    ("I would like to place a hold on a book", "borrow"),
    ("How long can I borrow books for", "borrow"),
    ("What is the maximum number of books I can borrow", "borrow"),
    ("I need to renew my borrowed book", "borrow"),
    ("Reserve a book for me", "borrow"),
    
    # Return Intent
    ("How do I return a book", "return"),
    ("Where is the return drop", "return"),
    ("Can I return books by mail", "return"),
    ("What happens if I return a book late", "return"),
    ("I need to return my borrowed books", "return"),
    ("Is there a late fee for overdue books", "return"),
    ("How do I renew my books", "return"),
    ("What is the late fee per day", "return"),
    
    # Library Info Intent
    ("Tell me about your library", "library_info"),
    ("What services does the library offer", "library_info"),
    ("Do you have WiFi access", "library_info"),
    ("Can I print documents here", "library_info"),
    ("Do you have study rooms", "library_info"),
    ("What databases are available", "library_info"),
    ("Can I access e-books", "library_info"),
    ("What events does the library host", "library_info"),
    
    # Account Intent
    ("How do I get a library card", "account"),
    ("I forgot my password", "account"),
    ("How do I renew my membership", "account"),
    ("Update my contact information", "account"),
    ("Check my account status", "account"),
    ("How do I update my email", "account"),
    ("I need to change my username", "account"),
    ("View my borrowing history", "account"),
    
    # Hours Intent
    ("What are your opening hours", "hours"),
    ("When does the library close", "hours"),
    ("Are you open on weekends", "hours"),
    ("What time do you open tomorrow", "hours"),
    ("Is the library open today", "hours"),
    ("Holiday hours", "hours"),
    ("Late night hours", "hours"),
    ("When are you open on Sundays", "hours"),
    
    # Identity Intent
    ("Who are you", "identity"),
    ("What is your name", "identity"),
    ("Are you a bot", "identity"),
    ("Tell me about yourself", "identity"),
    ("What can you help me with", "identity"),
    ("Who created you", "identity"),
    ("What are your capabilities", "identity"),
    ("Explain what you do", "identity")
]

# Separate into texts and labels
texts, labels = zip(*training_data)

print("=" * 60)
print("TRAINING DATA CREATED")
print("=" * 60)
print(f"Total training samples: {len(texts)}")
print("\nIntent distribution:")
for intent, count in Counter(labels).items():
    print(f"  - {intent}: {count} samples")

print("\nSample data points:")
for i in range(3):
    print(f'  Text: "{texts[i]}"')
    print(f"  Intent: {labels[i]}")
    print()


TRAINING DATA CREATED
Total training samples: 64

Intent distribution:
  - book_search: 8 samples
  - availability: 8 samples
  - borrow: 8 samples
  - return: 8 samples
  - library_info: 8 samples
  - account: 8 samples
  - hours: 8 samples
  - identity: 8 samples

Sample data points:
  Text: "I want to find a book about Python programming"
  Intent: book_search

  Text: "Can you search for books by author J.K. Rowling"
  Intent: book_search

  Text: "Find me books on machine learning"
  Intent: book_search



## Cell 3: Text Preprocessing - Tokenization

In [3]:
# Text Preprocessing Functions
def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    # 3. Tokenize
    tokens = word_tokenize(text)
    # 4. Remove stopwords
    stop_words = set(stopwords.words("english"))
    tokens = [t for t in tokens if t not in stop_words]
    # 5. Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

# Demonstrate preprocessing
print("=" * 60)
print("TEXT PREPROCESSING DEMO")
print("=" * 60)

sample_texts = [
    "I want to find a book about Python programming",
    "Are you open on weekends?",
    "Can I borrow Clean Code today?",
    "Who are you?"
]

for text in sample_texts:
    tokens = preprocess_text(text)
    print(f'\nOriginal: "{text}"')
    print(f"Tokens: {tokens}")
    print(f"Token count: {len(tokens)}")


TEXT PREPROCESSING DEMO

Original: "I want to find a book about Python programming"
Tokens: ['want', 'find', 'book', 'python', 'programming']
Token count: 5

Original: "Are you open on weekends?"
Tokens: ['open', 'weekend']
Token count: 2

Original: "Can I borrow Clean Code today?"
Tokens: ['borrow', 'clean', 'code', 'today']
Token count: 4

Original: "Who are you?"
Tokens: []
Token count: 0


## Cell 4: Feature Extraction - Bag of Words (BoW)

In [4]:
# Apply preprocessing to all training data
processed_texts = [" ".join(preprocess_text(t)) for t in texts]

# Create Bag-of-Words vectorizer
bow_vectorizer = CountVectorizer(max_features=100, min_df=1)
bow_matrix = bow_vectorizer.fit_transform(processed_texts)

print("=" * 60)
print("BAG-OF-WORDS FEATURE EXTRACTION")
print("=" * 60)

print(f"\nFeature Matrix Shape: {bow_matrix.shape}")
print(f"   - Rows (documents): {bow_matrix.shape[0]}")
print(f"   - Columns (vocabulary size): {bow_matrix.shape[1]}")

# Show vocabulary
vocabulary = bow_vectorizer.get_feature_names_out()
print(f"\nVocabulary (first 20 words):")
print(f"   {list(vocabulary[:20])}")

# Show a sample BoW vector
print(f"\nSample BoW vector (first document):")
sample_vector = bow_matrix[0].toarray()[0]
non_zero = [(vocabulary[i], sample_vector[i]) for i in range(len(sample_vector)) if sample_vector[i] > 0]
print(f"   {dict(non_zero)}")


BAG-OF-WORDS FEATURE EXTRACTION

Feature Matrix Shape: (64, 100)
   - Rows (documents): 64
   - Columns (vocabulary size): 100

Vocabulary (first 20 words):
   ['access', 'account', 'algorithm', 'author', 'available', 'book', 'borrow', 'borrowed', 'borrowing', 'bot', 'capability', 'card', 'catalog', 'catcher', 'change', 'check', 'checkout', 'clean', 'close', 'code']

Sample BoW vector (first document):
   {'book': np.int64(1), 'find': np.int64(1), 'programming': np.int64(1), 'python': np.int64(1), 'want': np.int64(1)}


## Cell 5: Feature Extraction - TF-IDF

In [5]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=100, min_df=1)
tfidf_matrix = tfidf_vectorizer.fit_transform(processed_texts)

print("=" * 60)
print("TF-IDF FEATURE EXTRACTION")
print("=" * 60)

print(f"\nTF-IDF Matrix Shape: {tfidf_matrix.shape}")

# Compare BoW vs TF-IDF
test_word = "book"
if test_word in vocabulary:
    idx = list(vocabulary).index(test_word)
    print(f"\nComparison for '{test_word}':")
    print(f"   BoW:   {bow_matrix[0, idx]}")
    print(f"   TF-IDF: {tfidf_matrix[0, idx]:.4f}")

# Show TF-IDF scores for first document
print(f"\nTF-IDF scores (first document, top features):")
sample_tfidf = tfidf_matrix[0].toarray()[0]
top_indices = sample_tfidf.argsort()[-5:][::-1]
for idx in top_indices:
    if sample_tfidf[idx] > 0:
        print(f"   {vocabulary[idx]}: {sample_tfidf[idx]:.4f}")


TF-IDF FEATURE EXTRACTION

TF-IDF Matrix Shape: (64, 100)

Comparison for 'book':
   BoW:   1
   TF-IDF: 0.2435

TF-IDF scores (first document, top features):
   python: 0.5238
   programming: 0.5238
   want: 0.4427
   find: 0.4427
   book: 0.2435


## Cell 6: Model Training - Intent Classification

In [6]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    tfidf_matrix, 
    labels, 
    test_size=0.2, 
    random_state=42,
    stratify=labels
)

# Train Naive Bayes classifier
classifier = MultinomialNB()
classifier.fit(X_train, y_train)

# Make predictions on test set
y_pred = classifier.predict(X_test)

print("=" * 60)
print("MODEL TRAINING - INTENT CLASSIFICATION")
print("=" * 60)

print(f"\nDataset Split:")
print(f"   Training samples: {X_train.shape[0]}")
print(f"   Testing samples: {X_test.shape[0]}")

print(f"\nModel: Multinomial Naive Bayes")
print(f"   Classes learned: {list(classifier.classes_)}")

# Show predictions vs actual
print(f"\nSample Predictions (Test Set):")
print("-" * 60)
for i in range(min(8, len(y_test))):
    status = "OK" if y_pred[i] == y_test[i] else "FAIL"
    print(f'[{status}] Input: "{texts[i][:40]}..."')
    print(f"   Predicted: {y_pred[i]} | Actual: {y_test[i]}")
    print()


MODEL TRAINING - INTENT CLASSIFICATION

Dataset Split:
   Training samples: 51
   Testing samples: 13

Model: Multinomial Naive Bayes
   Classes learned: [np.str_('account'), np.str_('availability'), np.str_('book_search'), np.str_('borrow'), np.str_('hours'), np.str_('identity'), np.str_('library_info'), np.str_('return')]

Sample Predictions (Test Set):
------------------------------------------------------------
[OK] Input: "I want to find a book about Python progr..."
   Predicted: book_search | Actual: book_search

[OK] Input: "Can you search for books by author J.K. ..."
   Predicted: return | Actual: return

[FAIL] Input: "Find me books on machine learning..."
   Predicted: library_info | Actual: hours

[OK] Input: "Search for the novel 1984..."
   Predicted: borrow | Actual: borrow

[OK] Input: "I am looking for science fiction books..."
   Predicted: identity | Actual: identity

[OK] Input: "Do you have any books about web developm..."
   Predicted: book_search | Actual: book_

## Cell 7: Model Evaluation - F1-Score Calculation

In [7]:
# Calculate evaluation metrics
accuracy = (y_pred == y_test).mean()
f1 = f1_score(y_test, y_pred, average="weighted")
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")

print("=" * 60)
print("MODEL EVALUATION - PERFORMANCE METRICS")
print("=" * 60)

print(f"\nOVERALL METRICS:")
print(f"   Accuracy:  {accuracy:.2%}")
print(f"   Precision: {precision:.2%}")
print(f"   Recall:    {recall:.2%}")
print(f"   F1-Score:  {f1:.2%}")

# Per-class breakdown
print(f"\nPER-CLASS F1-SCORES:")
print("-" * 60)
report = classification_report(y_test, y_pred)
print(report)

# Individual intent performance
from sklearn.metrics import precision_recall_fscore_support
p, r, f1_per_class, _ = precision_recall_fscore_support(y_test, y_pred, average=None)

print("\nIndividual Intent Performance:")
for i, cls in enumerate(classifier.classes_):
    print(f"   {cls:15} | Precision: {p[i]:.2f} | Recall: {r[i]:.2f} | F1: {f1_per_class[i]:.2f}")


MODEL EVALUATION - PERFORMANCE METRICS

OVERALL METRICS:
   Accuracy:  76.92%
   Precision: 75.64%
   Recall:    76.92%
   F1-Score:  73.85%

PER-CLASS F1-SCORES:
------------------------------------------------------------
              precision    recall  f1-score   support

     account       0.00      0.00      0.00         1
availability       1.00      1.00      1.00         1
 book_search       1.00      1.00      1.00         2
      borrow       0.67      1.00      0.80         2
       hours       1.00      0.50      0.67         2
    identity       0.50      1.00      0.67         1
library_info       0.50      0.50      0.50         2
      return       1.00      1.00      1.00         2

    accuracy                           0.77        13
   macro avg       0.71      0.75      0.70        13
weighted avg       0.76      0.77      0.74        13


Individual Intent Performance:
   account         | Precision: 0.00 | Recall: 0.00 | F1: 0.00
   availability    | Precision

C:\Users\mattw\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\mattw\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\mattw\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\mattw\anaconda3\Lib\site-packag

## Cell 8: Rule-Based Matching Demo

In [8]:
# Rule-based pattern matching
class RuleEngine:
    def __init__(self):
        self.rules = {
            r"(?:search|find|look for)\s+(.+?)(?:book|novel)": "book_search",
            r"(?:book|novel)\s+by\s+([A-Za-z\s]+)": "book_search",
            r"(?:is\s+)?(?:the\s+)?(?:book\s+)?(.+?)\s+(?:available|in\s+stock|on\s+shelf)": "availability",
            r"(?:what\s+)?(?:are\s+)?(?:your\s+)?(?:opening|closing)\s+hours": "hours",
            r"(?:forgot|lost|reset)\s+(?:my\s+)?password": "account",
            r"(?:how\s+do\s+I\s+)?return\s+(?:the\s+)?book": "return",
            r"(?:who\s+(?:are\s+)?you|what\s+is\s+your\s+name|tell\s+me\s+about\s+yourself)": "identity",
            r"(?:how\s+do\s+I\s+)?borrow|checkout|reserve|hold": "borrow",
        }
    
    def match(self, query):
        query_lower = query.lower()
        for pattern, intent in self.rules.items():
            if re.search(pattern, query_lower):
                return intent, pattern
        return None, None

rule_engine = RuleEngine()

print("=" * 60)
print("RULE-BASED MATCHING DEMO")
print("=" * 60)

test_queries = [
    "Find me a book about Python",
    "Is Harry Potter available?",
    "What are your opening hours?",
    "I forgot my password",
    "How do I return a book?",
    "Who are you?",
    "Can I borrow this book?",
]

print("\nRule Matching Results:")
print("-" * 60)
for query in test_queries:
    intent, pattern = rule_engine.match(query)
    status = "MATCH" if intent else "NO MATCH"
    print(f'\n[{status}] Query: "{query}"')
    if intent:
        print(f"   Intent: {intent}")
        print(f"   Pattern: {pattern}")
    else:
        print(f"   Will use ML classifier")


RULE-BASED MATCHING DEMO

Rule Matching Results:
------------------------------------------------------------

[MATCH] Query: "Find me a book about Python"
   Intent: book_search
   Pattern: (?:search|find|look for)\s+(.+?)(?:book|novel)

[MATCH] Query: "Is Harry Potter available?"
   Intent: availability
   Pattern: (?:is\s+)?(?:the\s+)?(?:book\s+)?(.+?)\s+(?:available|in\s+stock|on\s+shelf)

[MATCH] Query: "What are your opening hours?"
   Intent: hours
   Pattern: (?:what\s+)?(?:are\s+)?(?:your\s+)?(?:opening|closing)\s+hours

[MATCH] Query: "I forgot my password"
   Intent: account
   Pattern: (?:forgot|lost|reset)\s+(?:my\s+)?password

[NO MATCH] Query: "How do I return a book?"
   Will use ML classifier

[MATCH] Query: "Who are you?"
   Intent: identity
   Pattern: (?:who\s+(?:are\s+)?you|what\s+is\s+your\s+name|tell\s+me\s+about\s+yourself)

[MATCH] Query: "Can I borrow this book?"
   Intent: borrow
   Pattern: (?:how\s+do\s+I\s+)?borrow|checkout|reserve|hold


## Cell 9: Hybrid NLP + Rules Integration

In [9]:
class HybridIntentClassifier:
    def __init__(self, ml_classifier, vectorizer, rule_engine):
        self.classifier = ml_classifier
        self.vectorizer = vectorizer
        self.rule_engine = rule_engine
    
    def classify(self, query):
        # Step 1: Try rule-based matching first
        rule_intent, pattern = self.rule_engine.match(query)
        
        if rule_intent:
            return {
                "intent": rule_intent,
                "source": "rules",
                "confidence": 1.0,
                "pattern": pattern
            }
        
        # Step 2: Use ML classifier
        processed = " ".join(preprocess_text(query))
        features = self.vectorizer.transform([processed])
        
        intent = self.classifier.predict(features)[0]
        probabilities = self.classifier.predict_proba(features)[0]
        confidence = max(probabilities)
        
        return {
            "intent": intent,
            "source": "ml",
            "confidence": confidence
        }

hybrid_classifier = HybridIntentClassifier(classifier, tfidf_vectorizer, rule_engine)

print("=" * 60)
print("HYBRID CLASSIFIER (ML + RULES)")
print("=" * 60)

test_queries = [
    "Find me a book about Python",
    "Hello there!",
    "What time do you close?",
    "Tell me about databases",
    "I want to borrow 1984"
]

print("\nHybrid Classification Results:")
print("-" * 60)
for query in test_queries:
    result = hybrid_classifier.classify(query)
    print(f'\nQuery: "{query}"')
    print(f"   Intent: {result['intent']}")
    print(f"   Source: {result['source']}")
    print(f"   Confidence: {result['confidence']:.2%}")


HYBRID CLASSIFIER (ML + RULES)

Hybrid Classification Results:
------------------------------------------------------------

Query: "Find me a book about Python"
   Intent: book_search
   Source: rules
   Confidence: 100.00%

Query: "Hello there!"
   Intent: account
   Source: ml
   Confidence: 13.73%

Query: "What time do you close?"
   Intent: identity
   Source: ml
   Confidence: 14.44%

Query: "Tell me about databases"
   Intent: identity
   Source: ml
   Confidence: 19.99%

Query: "I want to borrow 1984"
   Intent: borrow
   Source: rules
   Confidence: 100.00%


## Cell 10: Response Generation Template

In [10]:
# Response templates
response_templates = {
    "book_search": [
        "I found several books matching your query.",
        "Let me search the catalog for you.",
        "Based on your search, here are some available books:",
    ],
    "availability": [
        "Let me check the availability for you.",
        "I will look up that book status right away.",
        "Checking our inventory...",
    ],
    "borrow": [
        "To borrow a book, you can use our online catalog.",
        "I can help you reserve that book.",
        "Our borrowing period is typically 2 weeks.",
    ],
    "return": [
        "You can return books to the main entrance drop box.",
        "Late fees are $0.25 per day.",
        "You can renew books online or at the circulation desk.",
    ],
    "library_info": [
        "Our library offers WiFi, study rooms, printing services.",
        "We provide various services including database access.",
        "The library has 5 study rooms and free WiFi.",
    ],
    "account": [
        "To get a library card, visit the front desk.",
        "You can reset your password through our online portal.",
        "I can help you with your account.",
    ],
    "hours": [
        "Our library is open Monday-Friday 9AM-8PM, Saturday 10AM-6PM.",
        "We are open on weekends!",
        "Check our holiday schedule.",
    ],
    "identity": [
        "I am the Library AI Assistant! I can help you find books.",
        "I am an AI chatbot designed to help you navigate library resources.",
        "Hello! I am here to help you with library services.",
    ],
}

import random

def generate_response(intent):
    templates = response_templates.get(intent, ["I am not sure how to help with that."])
    return random.choice(templates)

print("=" * 60)
print("RESPONSE GENERATION")
print("=" * 60)

print("\nSample Responses by Intent:")
print("-" * 60)
for intent in response_templates.keys():
    response = generate_response(intent)
    print(f"\nIntent: {intent}")
    print(f"   Response: {response}")


RESPONSE GENERATION

Sample Responses by Intent:
------------------------------------------------------------

Intent: book_search
   Response: Based on your search, here are some available books:

Intent: availability
   Response: Checking our inventory...

Intent: borrow
   Response: To borrow a book, you can use our online catalog.

Intent: return
   Response: You can renew books online or at the circulation desk.

Intent: library_info
   Response: Our library offers WiFi, study rooms, printing services.

Intent: account
   Response: I can help you with your account.

Intent: hours
   Response: Check our holiday schedule.

Intent: identity
   Response: I am an AI chatbot designed to help you navigate library resources.


## Cell 11: Complete End-to-End Pipeline Demo

In [11]:
def process_query(query):
    print(f"\n{'='*60}")
    print(f"USER QUERY: '{query}'")
    print(f"{'='*60}")
    
    # Step 1: Preprocessing
    print(f"\nSTEP 1: Preprocessing")
    tokens = preprocess_text(query)
    processed = " ".join(tokens)
    print(f"   Tokens: {tokens}")
    print(f'   Processed: "{processed}"')
    
    # Step 2: Classification
    print(f"\nSTEP 2: Intent Classification")
    result = hybrid_classifier.classify(query)
    print(f"   Detected Intent: {result['intent']}")
    print(f"   Classification Source: {result['source']}")
    print(f"   Confidence: {result['confidence']:.2%}")
    
    # Step 3: Response Generation
    print(f"\nSTEP 3: Response Generation")
    response = generate_response(result["intent"])
    print(f"   Response: {response}")
    
    print(f"\nFINAL OUTPUT")
    print(f"   {response}")
    
    return result

print("=" * 60)
print("COMPLETE END-TO-END PIPELINE DEMO")
print("=" * 60)

demo_queries = [
    "I want to find books about machine learning",
    "What are your opening hours?",
    "Who are you?",
    "Can I borrow The Great Gatsby?",
    "I forgot my password"
]

for query in demo_queries:
    process_query(query)


COMPLETE END-TO-END PIPELINE DEMO

USER QUERY: 'I want to find books about machine learning'

STEP 1: Preprocessing
   Tokens: ['want', 'find', 'book', 'machine', 'learning']
   Processed: "want find book machine learning"

STEP 2: Intent Classification
   Detected Intent: book_search
   Classification Source: ml
   Confidence: 27.96%

STEP 3: Response Generation
   Response: Let me search the catalog for you.

FINAL OUTPUT
   Let me search the catalog for you.

USER QUERY: 'What are your opening hours?'

STEP 1: Preprocessing
   Tokens: ['opening', 'hour']
   Processed: "opening hour"

STEP 2: Intent Classification
   Detected Intent: hours
   Classification Source: rules
   Confidence: 100.00%

STEP 3: Response Generation
   Response: We are open on weekends!

FINAL OUTPUT
   We are open on weekends!

USER QUERY: 'Who are you?'

STEP 1: Preprocessing
   Tokens: []
   Processed: ""

STEP 2: Intent Classification
   Detected Intent: identity
   Classification Source: rules
   Confidenc

## Cell 12: Summary & Key Takeaways

This notebook demonstrated the **inner workings** of our Intelligent Library Chat Assistant.

### What We Covered:
- **Environment Setup** - Library imports and versions
- **Training Data** - 64 labeled samples across 8 intents
- **Text Preprocessing** - Tokenization, stopword removal, lemmatization
- **Feature Extraction** - Bag-of-Words and TF-IDF with visible matrices
- **Model Training** - Naive Bayes classifier with 80/20 split
- **Evaluation** - F1-Score, Precision, Recall with actual numbers
- **Rule Engine** - Pattern-based matching for common queries
- **Hybrid System** - Combining ML and Rules for better accuracy
- **Response Generation** - Template-based responses
- **End-to-End Demo** - Complete pipeline with outputs at each step

### Evaluation Metrics Summary:
- **Accuracy**: How often the model is correct overall
- **Precision**: Of predicted positives, how many are correct
- **Recall**: Of actual positives, how many did we find
- **F1-Score**: Balance between precision and recall

### Architecture Benefits:
1. **Hybrid approach** provides both flexibility (ML) and precision (Rules)
2. **TF-IDF** helps distinguish important words from common words
3. **Rule patterns** ensure reliable handling of common queries
4. **Evaluation metrics** allow continuous improvement

---
*This notebook aligns with Chapter 3: Architecture, Flowchart, and Evaluation Metrics from your project documentation.*

In [12]:
# Final Summary Statistics
print("=" * 60)
print("FINAL SUMMARY - KEY METRICS")
print("=" * 60)
print()
print("LIBRARY CHATBOT EVALUATION RESULTS")
print("-" * 60)
print(f"Training Samples:     {len(texts)}")
print(f"Number of Intents:    {len(set(labels))}")
print(f"Vocabulary Size:      {len(vocabulary)}")
print("-" * 60)
print(f"Accuracy:             {accuracy:.2%}")
print(f"Precision (weighted): {precision:.2%}")
print(f"Recall (weighted):    {recall:.2%}")
print(f"F1-Score (weighted):  {f1:.2%}")
print("-" * 60)
print("System Type:           Hybrid (ML + Rules)")
print("ML Model:              Multinomial Naive Bayes")
print("Feature Method:        TF-IDF")
print()
print("Notebook Complete!")


FINAL SUMMARY - KEY METRICS

LIBRARY CHATBOT EVALUATION RESULTS
------------------------------------------------------------
Training Samples:     64
Number of Intents:    8
Vocabulary Size:      100
------------------------------------------------------------
Accuracy:             76.92%
Precision (weighted): 75.64%
Recall (weighted):    76.92%
F1-Score (weighted):  73.85%
------------------------------------------------------------
System Type:           Hybrid (ML + Rules)
ML Model:              Multinomial Naive Bayes
Feature Method:        TF-IDF

Notebook Complete!
